### Dataset and Task Metadata

In [ ]:
from tokenize import group

from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="sepsis_prediction_1m",
    version_from_unique_name="sepsis_prediction",
    version_comment="""
We build the recommended single train/test split (250k test rows) and sub-sample it to at most 1M train and 250k test rows with `curation_recommendations.subsample_split_to_budget`, keeping whole patients.
""",
    # Same as the full-size `sepsis_prediction` dataset
    dataset_year="2019",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/salikhussaini49/prediction-of-sepsis", # alt https://physionet.org/content/challenge-2019/1.0.0/
    download_description="""
We download the data from Kaggle as the link for the data from the original PhysioNet challenge seems to be down.

kaggle datasets download -d salikhussaini49/prediction-of-sepsis && unzip prediction-of-sepsis.zip all_files && rm prediction-of-sepsis.zip
mkdir -p local-data-warehouse/sepsis_prediction && mv all_files local-data-warehouse/sepsis_prediction/
""",
    # References
    academic_reference_bibtex="""@article{reyna2020early,
  title={Early prediction of sepsis from clinical data: the PhysioNet/Computing in Cardiology Challenge 2019},
  author={Reyna, Matthew A and Josef, Christopher S and Jeter, Russell and Shashikumar, Supreeth P and Westover, M Brandon and Nemati, Shamim and Clifford, Gari D and Sharma, Ashish},
  journal={Critical care medicine},
  volume={48},
  number={2},
  pages={210--217},
  year={2020},
  publisher={LWW}
}
""",
    academic_reference_bibtex_key="reyna2020early",
    license="ODC Open Database License", # CC BY-NC-SA 4.0 on Kaggle...
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We start with all files from Kaggle.

The original data has one file per user that was already preprocessed to one .csv file by the competition creators. Here we start with the preprocessed single .csv file. The data is non-IID in nature based on the groups of patients from different hospitals. The data that is grouped per patient ("Patient_ID"). These groups are also temporal in nature, but this temporal dependency is irrelevant as teh task is to predicts for one full patient (i.e., no refitting given patient information).
In the original competition, one had to predict for unseen patients from the existing hospitals and also for a new hidden hospital. In the public data, we only have two hospitals, (A) and (B). Given the limited data, we decide not to simulate a domain shift as we could not "train" for domain shift. Thus, we simulate only a normal grouped non-IID scenario. That is, we use all patients from both hospitals for training and testing, but ensure that the splits are grouped by patient ID. Thus, we simulate what would happen if someone trains a model on data from two hospitals and uses this to predict for other patients from these hospitals. This decision is also amplified by the large gap in performance for the unseen hospital in the competition (Report, Table 3), clearly pointing to a domain shift that is out-of-scope for this task. Note, we do not have temporal information of the order of patients, thus, we cannot simulate to only predict for patients "from the future". We believe this does not introduce any data leakage for this dataset.

- Patients with an ID larger than 100_000 are from hospital (B), while patients with an ID smaller than 100_000 are from hospital (A). We add this indicator into our data and then reset the Patient_IDs to be continuous increasing integers.
- The data consists of a lot of missing values due to missing measurements.
- Note, hour is a reset time index per patient. ICULOS is similar, but with an offset. We keep both as this can show that we have truncated data for a patient.
- We reverse the ordinal encoding of Gender.
- We mark features as categorical where appropriate.
- For each patient, we predict the sepsis status over time. So it is more or less a transformed survival task (also in the original challenge).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="SepsisLabel",
    problem_type="binary_classification",
    objective_metric_name="PhysioNet2019UtilityFunction", # https://github.com/physionetchallenges/evaluation-2019/blob/master/evaluate_sepsis_score.py
    stratify_on="SepsisLabel",
    group_on="Patient_ID",
    group_time_on="Hour",
    group_labels="per_sample",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "all_files" / "Dataset.csv")
print("Loaded data shape:", df.shape)

# Add Hospital Indicator
df["Hospital"] = "Hospital_A"
df.loc[df["Patient_ID"] > 100_000, "Hospital"] = "Hospital_B"
# Reset Patient IDs to be continuous (after remapping, IDs higher than 20k are from Hospital B)
codes, _ = pd.factorize(df["Patient_ID"])
df["Patient_ID"] = codes

df["Gender"] = df["Gender"].replace({0: "Female", 1: "Male"})

as_cat_type = ["Hospital", "SepsisLabel", "Patient_ID", "Gender", "Unit1", "Unit2"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.drop(columns=["Unnamed: 0"])

# Remove order of patients so that method figure this our themselves (remove order by "Patient_ID", "Hour")
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
    duplicate_column_check=False, # We know the data has unique columns
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)

# -- For Grouped Non-IID data
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
    random_state=split_random_state,
)

# Sub-sample the single train/test split to the row budget (1M train / 250k test rows), keeping whole patients.
train_idx, test_idx = splits[0][0]
df, train_idx, test_idx = curation_recommendations.subsample_split_to_budget(
    df=df,
    train_idx=train_idx,
    test_idx=test_idx,
    group_on=task_mold.group_on,
    stratify_on=task_mold.stratify_on,
    random_state=split_random_state,
)
splits = {0: {0: (train_idx, test_idx)}}
print(f"Final train size: {len(train_idx):,}, final test size: {len(test_idx):,}")
print(f"Final train groups: {df.iloc[train_idx][task_mold.group_on].nunique():,}, final test groups: {df.iloc[test_idx][task_mold.group_on].nunique():,}")

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Recommended single grouped train/test split, sub-sampled to the row budget of 1M train and 250k test rows, keeping whole patients.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()